# Build county heat-advisory replay data

This notebook creates one row per **Iowa county × crop × replay year** using only weather and drought observations available through a crop-specific advisory date. It produces `advisory_df`, the DataFrame for the dashboard.

- Corn cutoff: July 15
- Soybean cutoff: August 1
- `data_mode` is intentionally `historical_replay`; this is not a live forecast.
- The score is a transparent weighted prototype. Freeze its settings before evaluating 2019–2025.

In [10]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Works whether Jupyter starts from the repository root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run this notebook from the repository root or its notebooks/ folder.')

sys.path.insert(0, str(ROOT))
from src.data_loader import load_daily_weather, load_weekly_drought, load_soil


In [11]:
# --- Configuration: change these deliberately, then keep them fixed for evaluation. ---
STATE = 'IA'
REPLAY_YEARS = list(range(2000, 2027))  # range end is exclusive; includes 2026

# The daily/weekly source data are aggregated only through these dates.
ADVISORY_CUTOFFS = {
    'CORN': (7, 15),       # July 15
    'SOYBEANS': (8, 1),   # August 1
}

# Each crop's supplied season begins May 1.
SEASON_START = (5, 1)

# Transparent weighted-score formula. Components are all capped at 1.0.
HOT35_CAP_DAYS = 8
D2PLUS_CAP_WEEKS = 6
WEIGHT_HEAT = 0.60
WEIGHT_DROUGHT = 0.30
WEIGHT_SOIL = 0.10
FORMULA_VERSION = 'weighted_v1'

assert np.isclose(WEIGHT_HEAT + WEIGHT_DROUGHT + WEIGHT_SOIL, 1.0)


In [19]:
# Load source panels. FIPS stays as a five-character string throughout.
weather = load_daily_weather().copy()
drought = load_weekly_drought().copy()
soil = load_soil().copy()

for frame in (weather, drought, soil):
    frame['fips'] = frame['fips'].astype(str).str.zfill(5)

weather['date'] = pd.to_datetime(weather['date'])
drought['map_date'] = pd.to_datetime(drought['map_date'])

ia_weather = weather.loc[weather['state_alpha'].eq(STATE)].copy()
ia_drought = drought.loc[drought['state_alpha'].eq(STATE)].copy()
ia_soil = soil.loc[soil['state_alpha'].eq(STATE)].copy()

print(f'Daily weather rows: {len(ia_weather):,}')
print(f'Weekly drought rows: {len(ia_drought):,}')
print(f'Iowa soil counties: {ia_soil.fips.nunique()}')
print( weather.tail() ) 


Daily weather rows: 1,005,048
Weekly drought rows: 138,402
Iowa soil counties: 99
          fips       date state_alpha    pcpn_mm     tmax_c    tmin_c
2568451  31185 2026-10-13          NE   0.000000  31.873276  8.771649
2568452  31185 2026-10-14          NE   0.000000  14.410212  5.991863
2568453  31185 2026-10-15          NE  21.097404  13.918095  7.509139
2568454  31185 2026-10-16          NE   8.280983  13.205034  5.193041
2568455  31185 2026-10-17          NE   0.000000  12.857456 -0.630617


In [13]:
# Soil capacity is static by county. Classify the 99 Iowa counties into storage tertiles.
soil_context = ia_soil[['fips', 'aws_100cm_mm']].dropna(subset=['aws_100cm_mm']).drop_duplicates('fips').copy()
soil_context['soil_storage_group'] = pd.qcut(
    soil_context['aws_100cm_mm'].rank(method='first'),
    q=3,
    labels=['LOW', 'MEDIUM', 'HIGH'],
).astype(str)
soil_context['soil_vulnerability_component'] = soil_context['soil_storage_group'].map({
    'LOW': 1.0,
    'MEDIUM': 0.5,
    'HIGH': 0.0,
})

# County names live in the weekly drought panel. Use its latest record as a county lookup.
county_lookup = (
    ia_drought.sort_values('map_date')
    .drop_duplicates('fips', keep='last')
    [['fips', 'county_name', 'state_alpha']]
)

soil_context.groupby('soil_storage_group', observed=True)['aws_100cm_mm'].agg(['count', 'min', 'median', 'max'])


,count,min,median,max
soil_storage_group,,,,
HIGH,33,182.721663,189.884001,205.757359
LOW,33,154.245463,166.993439,173.808059
MEDIUM,33,174.204646,178.999037,182.695271


In [14]:
# Turn weekly Drought Monitor category percentages into the project's 0–5 severity scale.
# 0 = no drought; 1 = D0; 2 = D1; 3 = D2; 4 = D3; 5 = D4.
drought_scored = ia_drought.copy()
drought_scored['weekly_severity'] = 0
for severity, column in enumerate(['D0', 'D1', 'D2', 'D3', 'D4'], start=1):
    drought_scored.loc[drought_scored[column] > 1, 'weekly_severity'] = severity

# A D2+ week occurs when severe, extreme, or exceptional drought covers >1% of the county.
drought_scored['d2plus_week'] = (
    drought_scored[['D2', 'D3', 'D4']].sum(axis=1) > 1
).astype(int)


In [21]:
# Build observed-to-date weather and drought features for every replay year and crop.
feature_parts = []

for crop, (month, day) in ADVISORY_CUTOFFS.items():
    for year in REPLAY_YEARS:
        season_start = pd.Timestamp(year=year, month=SEASON_START[0], day=SEASON_START[1])
        as_of_date = pd.Timestamp(year=year, month=month, day=day)

        weather_window = ia_weather.loc[
            ia_weather['date'].between(season_start, as_of_date)
        ].copy()
        weather_features = (
            weather_window.assign(hot35_day=weather_window['tmax_c'].ge(35).astype(int))
            .groupby('fips', as_index=False)
            .agg(
                hot35_days_to_date=('hot35_day', 'sum'),
                weather_through_date=('date', 'max'),
            )
        )

        drought_window = drought_scored.loc[
            drought_scored['map_date'].between(season_start, as_of_date)
        ].copy()
        drought_features = (
            drought_window.groupby('fips', as_index=False)
            .agg(
                d2plus_weeks_to_date=('d2plus_week', 'sum'),
                max_drought_severity_to_date=('weekly_severity', 'max'),
                drought_through_date=('map_date', 'max'),
            )
        )

        features = weather_features.merge(drought_features, on='fips', how='outer', validate='one_to_one')
        features['crop'] = crop
        features['year'] = year
        features['as_of_date'] = as_of_date
        feature_parts.append(features)

observed_features = pd.concat(feature_parts, ignore_index=True)
assert not observed_features.duplicated(['fips', 'crop', 'year']).any()
observed_features.head()
observed_features.tail()


,fips,hot35_days_to_date,weather_through_date,d2plus_weeks_to_date,max_drought_severity_to_date,drought_through_date,crop,year,as_of_date
5143,19189,1,2025-08-01,0,1,2025-07-29,SOYBEANS,2025,2025-08-01
5144,19191,0,2025-08-01,0,1,2025-07-29,SOYBEANS,2025,2025-08-01
5145,19193,2,2025-08-01,0,2,2025-07-29,SOYBEANS,2025,2025-08-01
5146,19195,0,2025-08-01,0,1,2025-07-29,SOYBEANS,2025,2025-08-01
5147,19197,0,2025-08-01,0,0,2025-07-29,SOYBEANS,2025,2025-08-01


In [16]:
# Add identifiers and static soil context, then calculate all formula components.
advisory_df = (
    observed_features
    .merge(county_lookup, on='fips', how='left', validate='many_to_one')
    .merge(soil_context, on='fips', how='left', validate='many_to_one')
)

# Do not turn missing source data into zero. Incomplete rows receive no score.
required = [
    'hot35_days_to_date',
    'd2plus_weeks_to_date',
    'max_drought_severity_to_date',
    'aws_100cm_mm',
    'soil_vulnerability_component',
]
complete = advisory_df[required].notna().all(axis=1)

advisory_df['heat_exposure_component'] = (advisory_df['hot35_days_to_date'] / HOT35_CAP_DAYS).clip(0, 1)
advisory_df['drought_duration_component'] = (advisory_df['d2plus_weeks_to_date'] / D2PLUS_CAP_WEEKS).clip(0, 1)
advisory_df['drought_severity_component'] = ((advisory_df['max_drought_severity_to_date'] - 2).clip(lower=0) / 3).clip(0, 1)
advisory_df['drought_stress_component'] = (
    0.65 * advisory_df['drought_duration_component']
    + 0.35 * advisory_df['drought_severity_component']
)
advisory_df['heat_risk_score'] = 100 * (
    WEIGHT_HEAT * advisory_df['heat_exposure_component']
    + WEIGHT_DROUGHT * advisory_df['drought_stress_component']
    + WEIGHT_SOIL * advisory_df['soil_vulnerability_component']
)

score_columns = [
    'heat_exposure_component', 'drought_duration_component',
    'drought_severity_component', 'drought_stress_component',
    'heat_risk_score',
]
advisory_df.loc[~complete, score_columns] = np.nan
advisory_df['heat_risk_score'] = advisory_df['heat_risk_score'].round(1)
advisory_df['risk_level'] = np.select(
    [advisory_df['heat_risk_score'].ge(60), advisory_df['heat_risk_score'].ge(35)],
    ['HIGH', 'WATCH'],
    default='LOW',
)
advisory_df.loc[~complete, 'risk_level'] = 'UNAVAILABLE'
advisory_df['advisory_action'] = np.select(
    [
        advisory_df['risk_level'].eq('HIGH'),
        advisory_df['risk_level'].eq('WATCH'),
        advisory_df['risk_level'].eq('LOW'),
    ],
    [
        'Prioritize extension outreach',
        'Monitor next weekly update',
        'Continue routine monitoring',
    ],
    default='Check source-data coverage',
)
advisory_df['data_mode'] = np.where(advisory_df['year'].gt(2025), 'current', 'historical_replay')
advisory_df['formula_version'] = FORMULA_VERSION


In [20]:
# Keep the requested dashboard fields first. `year` is retained as a replay/evaluation helper.
requested_columns = [
    'fips', 'county_name', 'state_alpha', 'crop', 'year',
    'as_of_date', 'weather_through_date', 'drought_through_date', 'data_mode',
    'hot35_days_to_date', 'd2plus_weeks_to_date', 'max_drought_severity_to_date',
    'aws_100cm_mm', 'soil_storage_group',
    'heat_exposure_component', 'drought_duration_component',
    'drought_severity_component', 'drought_stress_component',
    'soil_vulnerability_component', 'heat_risk_score', 'risk_level',
    'advisory_action', 'formula_version',
]
advisory_df = advisory_df[requested_columns].sort_values(['year', 'crop', 'fips']).reset_index(drop=True)

for column in ['as_of_date', 'weather_through_date', 'drought_through_date']:
    advisory_df[column] = pd.to_datetime(advisory_df[column]).dt.strftime('%Y-%m-%d')

assert advisory_df['fips'].str.len().eq(5).all()
assert not advisory_df.duplicated(['fips', 'crop', 'year']).any()

print(f'Rows: {len(advisory_df):,}')
print(f'Rows with unavailable data: {(advisory_df.risk_level == "UNAVAILABLE").sum():,}')
display(advisory_df.head())
display(advisory_df.tail())


Rows: 5,148
Rows with unavailable data: 0


,fips,county_name,state_alpha,crop,year,as_of_date,weather_through_date,drought_through_date,data_mode,hot35_days_to_date,...,soil_storage_group,heat_exposure_component,drought_duration_component,drought_severity_component,drought_stress_component,soil_vulnerability_component,heat_risk_score,risk_level,advisory_action,formula_version
0,19001,Adair County,IA,CORN,2000,2000-07-15,2000-07-15,2000-07-11,historical_replay,0,...,MEDIUM,0.0,1.0,0.666667,0.883333,0.5,31.5,LOW,Continue routine monitoring,weighted_v1
1,19003,Adams County,IA,CORN,2000,2000-07-15,2000-07-15,2000-07-11,historical_replay,0,...,MEDIUM,0.0,1.0,0.666667,0.883333,0.5,31.5,LOW,Continue routine monitoring,weighted_v1
2,19005,Allamakee County,IA,CORN,2000,2000-07-15,2000-07-15,2000-07-11,historical_replay,0,...,LOW,0.0,0.0,0.000000,0.000000,1.0,10.0,LOW,Continue routine monitoring,weighted_v1
3,19007,Appanoose County,IA,CORN,2000,2000-07-15,2000-07-15,2000-07-11,historical_replay,0,...,LOW,0.0,1.0,0.333333,0.766667,1.0,33.0,LOW,Continue routine monitoring,weighted_v1
4,19009,Audubon County,IA,CORN,2000,2000-07-15,2000-07-15,2000-07-11,historical_replay,0,...,HIGH,0.0,1.0,0.333333,0.766667,0.0,23.0,LOW,Continue routine monitoring,weighted_v1


,fips,county_name,state_alpha,crop,year,as_of_date,weather_through_date,drought_through_date,data_mode,hot35_days_to_date,...,soil_storage_group,heat_exposure_component,drought_duration_component,drought_severity_component,drought_stress_component,soil_vulnerability_component,heat_risk_score,risk_level,advisory_action,formula_version
5143,19189,Winnebago County,IA,SOYBEANS,2025,2025-08-01,2025-08-01,2025-07-29,historical_replay,1,...,HIGH,0.125,0.0,0.0,0.0,0.0,7.5,LOW,Continue routine monitoring,weighted_v1
5144,19191,Winneshiek County,IA,SOYBEANS,2025,2025-08-01,2025-08-01,2025-07-29,historical_replay,0,...,MEDIUM,0.000,0.0,0.0,0.0,0.5,5.0,LOW,Continue routine monitoring,weighted_v1
5145,19193,Woodbury County,IA,SOYBEANS,2025,2025-08-01,2025-08-01,2025-07-29,historical_replay,2,...,HIGH,0.250,0.0,0.0,0.0,0.0,15.0,LOW,Continue routine monitoring,weighted_v1
5146,19195,Worth County,IA,SOYBEANS,2025,2025-08-01,2025-08-01,2025-07-29,historical_replay,0,...,LOW,0.000,0.0,0.0,0.0,1.0,10.0,LOW,Continue routine monitoring,weighted_v1
5147,19197,Wright County,IA,SOYBEANS,2025,2025-08-01,2025-08-01,2025-07-29,historical_replay,0,...,LOW,0.000,0.0,0.0,0.0,1.0,10.0,LOW,Continue routine monitoring,weighted_v1


In [18]:
# Optional dashboard export. Run this cell after reviewing advisory_df.
OUTPUT_DIR = ROOT / 'data' / 'derived'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

advisory_df.to_csv(OUTPUT_DIR / 'county_heat_advisory.csv', index=False)
advisory_df.to_parquet(OUTPUT_DIR / 'county_heat_advisory.parquet', index=False)

# The React app reads this public copy at runtime. Re-running this notebook refreshes the dashboard data too.
DASHBOARD_CSV = ROOT / 'web' / 'public' / 'data' / 'county_heat_advisory.csv'
advisory_df.to_csv(DASHBOARD_CSV, index=False)

print(f'Saved data outputs to {OUTPUT_DIR}')
print(f'Refreshed dashboard data: {DASHBOARD_CSV}')


Saved to /Users/hesanders23/Desktop/MSCC Hackathon/DSU-Hackathon-2026-Team-AHJ/data/derived
